# SKU-Node Correlation Analysis

Exploratory analysis of correlations between pricing, inventory, and sales at the SKU-Node-Date level.

**Business questions:**
- How do NLC price changes affect subsequent sales volume?
- What factors most influence sales at the SKU-Node level?
- What margin levels maximize total revenue/profit?

## 1. Parameters & Setup

In [ ]:
# === PARAMETERS ===
ANALYSIS_DAYS = 30
END_DATE = "2026-03-25"
TOP_SKU_PCT = 0.90
SKU_SALES_LOOKBACK_DAYS = 60
ROLLING_WINDOW = 7
INVENTORY_SAMPLE_FREQ = "4D"
WAREHOUSE_ADDRESSES_PATH = r"C:\Users\valen\Desktop\WalmartPricing\Warehouse Addresses 03-25-2026 01-43-16 PM.csv"
ROLLBACKS_PATH = None  # Set to rollbacks Excel path to exclude rollback SKUs

In [ ]:
import sys
import os
import logging
import warnings
from datetime import timedelta

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Add project root to path
sys.path.insert(0, os.path.abspath(".."))

from src.data.loader import DataLoader
from src.adapters.module_loader import load_yaml, ensure_modules_path

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

In [ ]:
# Derived date boundaries
end_dt = pd.to_datetime(END_DATE)
start_dt = end_dt - pd.Timedelta(days=ANALYSIS_DAYS - 1)
warmup_dt = start_dt - pd.Timedelta(days=ROLLING_WINDOW)
sku_filter_start_dt = end_dt - pd.Timedelta(days=SKU_SALES_LOOKBACK_DAYS)

START_DATE = start_dt.strftime("%Y-%m-%d")
WARMUP_DATE = warmup_dt.strftime("%Y-%m-%d")
SKU_FILTER_START = sku_filter_start_dt.strftime("%Y-%m-%d")

print(f"Analysis window: {START_DATE} to {END_DATE}")
print(f"Warmup start (for rolling): {WARMUP_DATE}")
print(f"SKU filter lookback start: {SKU_FILTER_START}")

In [ ]:
# Initialize loader and configs
loader = DataLoader()
nlc_config = load_yaml("nlc_model.yaml")

## 2. SKU Filtering (top 90% by qty sold)

In [ ]:
# Load 2-month sales for SKU filtering
df_sales_filter = loader.load("dw_walmart_sales", start_date=SKU_FILTER_START)
print(f"Sales rows (last {SKU_SALES_LOOKBACK_DAYS} days): {len(df_sales_filter):,}")

In [ ]:
# Identify top SKUs by cumulative quantity sold
df_sku_qty = (
    df_sales_filter.groupby("sku")["quantity"]
    .sum()
    .reset_index()
    .rename(columns={"quantity": "total_qty"})
    .sort_values("total_qty", ascending=False)
    .reset_index(drop=True)
)
df_sku_qty["cum_qty"] = df_sku_qty["total_qty"].cumsum()
df_sku_qty["cum_pct"] = df_sku_qty["cum_qty"] / df_sku_qty["total_qty"].sum()

# Include the first SKU that crosses the threshold
cutoff_idx = (df_sku_qty["cum_pct"] <= TOP_SKU_PCT).sum()
top_skus = set(df_sku_qty.iloc[: cutoff_idx + 1]["sku"].tolist())

print(f"Total unique SKUs with sales: {df_sku_qty.shape[0]:,}")
print(f"Top {TOP_SKU_PCT:.0%} SKUs selected: {len(top_skus):,}")
print(f"Coverage: {df_sku_qty.iloc[:cutoff_idx + 1]['total_qty'].sum() / df_sku_qty['total_qty'].sum():.1%}")

In [ ]:
# Optionally exclude rollback SKUs
if ROLLBACKS_PATH:
    df_rollbacks = loader.load("rollbacks", rollbacks_path=ROLLBACKS_PATH)
    df_rollbacks["End date"] = pd.to_datetime(df_rollbacks["End date"])
    active_rollbacks = df_rollbacks[df_rollbacks["End date"] > start_dt]["Product Code"].unique()
    before = len(top_skus)
    top_skus -= set(active_rollbacks)
    print(f"Excluded {before - len(top_skus)} rollback SKUs, {len(top_skus)} remaining")
else:
    print("Rollback exclusion skipped (ROLLBACKS_PATH not set)")

## 3. Load Daily Sales

In [ ]:
# Load sales for analysis window + 7-day warmup
df_sales_raw = loader.load("dw_walmart_sales", start_date=WARMUP_DATE)
df_sales_raw["order_date"] = pd.to_datetime(df_sales_raw["order_date"])

# Filter to top SKUs and analysis+warmup window
df_sales_raw = df_sales_raw[
    (df_sales_raw["sku"].isin(top_skus))
    & (df_sales_raw["order_date"] <= end_dt)
].copy()

print(f"Sales rows after filtering: {len(df_sales_raw):,}")

In [ ]:
# Aggregate to SKU-Node-Date level
df_sales_agg = (
    df_sales_raw
    .groupby(["sku", "externalwarehouseid", "order_date"])
    .agg(
        qty_sold=("quantity", "sum"),
        revenue=("total_inv_amount", "sum"),
        profit=("profit", "sum"),
    )
    .reset_index()
    .rename(columns={"externalwarehouseid": "node", "order_date": "date"})
)
df_sales_agg["node"] = df_sales_agg["node"].astype(str)

# Valid SKU-Nodes: any with at least 1 sale in the full window
sku_nodes = df_sales_agg[["sku", "node"]].drop_duplicates()
print(f"Unique SKU-Nodes with sales: {len(sku_nodes):,}")
print(f"Daily aggregated rows: {len(df_sales_agg):,}")

## 4. Build Date Scaffold

In [ ]:
# Full date range including warmup
all_dates = pd.date_range(warmup_dt, end_dt, freq="D")
df_dates = pd.DataFrame({"date": all_dates})

# Cross-join: every SKU-Node x every date
scaffold = sku_nodes.merge(df_dates, how="cross")
print(f"Scaffold shape: {scaffold.shape[0]:,} rows ({len(sku_nodes):,} SKU-Nodes x {len(all_dates)} days)")

# Left-join sales
scaffold = scaffold.merge(df_sales_agg, on=["sku", "node", "date"], how="left")
scaffold["qty_sold"] = scaffold["qty_sold"].fillna(0)
scaffold["revenue"] = scaffold["revenue"].fillna(0)
scaffold["profit"] = scaffold["profit"].fillna(0)

print(f"Non-zero sales days: {(scaffold['qty_sold'] > 0).sum():,} / {len(scaffold):,}")

## 5. DSV History (Cost to Walmart)

In [ ]:
import re

# List all DSV files and parse dates
dsv_config = loader.get_source_config("walmart_dsv_folder")
dsv_files = loader.google.get_folder_files(dsv_config["id"])

dsv_files["parsed_date"] = dsv_files["Name"].apply(
    lambda x: re.search(r"\d{4}-\d{2}-\d{2}", str(x))
)
dsv_files = dsv_files[dsv_files["parsed_date"].notna()].copy()
dsv_files["dsv_date"] = dsv_files["parsed_date"].apply(lambda m: pd.to_datetime(m.group()))
dsv_files = dsv_files.drop(columns=["parsed_date"])

# Filter to files relevant to our window (warmup_dt - 30 days buffer to END_DATE)
buffer_start = warmup_dt - pd.Timedelta(days=30)
dsv_files = dsv_files[
    (dsv_files["dsv_date"] >= buffer_start) & (dsv_files["dsv_date"] <= end_dt)
].sort_values("dsv_date")

# Deduplicate: keep only the LATEST file per date (later upload supersedes)
dsv_files = dsv_files.sort_values("Name").groupby("dsv_date").last().reset_index()
dsv_files = dsv_files.sort_values("dsv_date")

print(f"DSV files in range (deduplicated): {len(dsv_files)}")
print(dsv_files[["Name", "dsv_date"]].to_string(index=False))


In [ ]:
# Load each DSV file, filtering to top SKUs immediately to manage memory
# Each raw file has ~2M rows; filtering to top 6K SKUs reduces to ~125K each
dsv_snapshots = []
for idx, (_, row) in enumerate(dsv_files.iterrows()):
    df_dsv_i = loader.google.get_file_as_df(
        row["ID"], "csv",
        read_cols=["SKU", "Price", "Source"],
        dtype={"SKU": str, "Price": float, "Source": str},
    )
    # Filter to top SKUs immediately to save memory
    df_dsv_i = df_dsv_i[df_dsv_i["SKU"].isin(top_skus)].copy()
    df_dsv_i = df_dsv_i.drop_duplicates(subset=["SKU", "Source"], keep="first")
    df_dsv_i["dsv_date"] = row["dsv_date"]
    dsv_snapshots.append(df_dsv_i)
    if (idx + 1) % 10 == 0 or idx == 0:
        logger.info("Loaded DSV %d/%d: %s (%d rows after SKU filter)",
                     idx + 1, len(dsv_files), row["Name"], len(df_dsv_i))

df_dsv_all = pd.concat(dsv_snapshots, ignore_index=True)
df_dsv_all = df_dsv_all.rename(columns={"SKU": "sku", "Price": "cost_to_walmart", "Source": "source"})
df_dsv_all["source"] = df_dsv_all["source"].astype(str)

print(f"Total DSV rows (filtered to top SKUs): {len(df_dsv_all):,}")
print(f"Unique DSV dates: {df_dsv_all['dsv_date'].nunique()}")


In [ ]:
# For each scaffold row, assign cost_to_walmart from the most recent DSV
# Strategy: for each analysis date, find the latest DSV date <= that date

# Step 1: Node-specific prices (source = node identifier)
df_dsv_node = df_dsv_all[df_dsv_all["source"] != "nan"].copy()
df_dsv_node = df_dsv_node.rename(columns={"source": "node"})
df_dsv_node = df_dsv_node.sort_values("dsv_date")

# Step 2: National prices (source = nan/null) as fallback
df_dsv_national = df_dsv_all[df_dsv_all["source"] == "nan"].copy()
df_dsv_national = df_dsv_national[["sku", "cost_to_walmart", "dsv_date"]].sort_values("dsv_date")

# merge_asof for node-specific prices
scaffold = scaffold.sort_values("date")
df_dsv_node = df_dsv_node.sort_values("dsv_date")

scaffold = pd.merge_asof(
    scaffold.sort_values("date"),
    df_dsv_node[["sku", "node", "cost_to_walmart", "dsv_date"]].sort_values("dsv_date"),
    left_on="date",
    right_on="dsv_date",
    by=["sku", "node"],
    direction="backward",
    suffixes=("", "_dsv"),
)

# Fill NaN cost_to_walmart with national price fallback
scaffold_missing = scaffold[scaffold["cost_to_walmart"].isna()].copy()
if len(scaffold_missing) > 0:
    scaffold_missing = scaffold_missing.drop(columns=["cost_to_walmart", "dsv_date"])
    scaffold_missing = pd.merge_asof(
        scaffold_missing.sort_values("date"),
        df_dsv_national.rename(columns={"cost_to_walmart": "cost_to_walmart_nat", "dsv_date": "dsv_date_nat"}),
        left_on="date",
        right_on="dsv_date_nat",
        by="sku",
        direction="backward",
    )
    # Update scaffold with national fallback
    scaffold.loc[scaffold["cost_to_walmart"].isna(), "cost_to_walmart"] = (
        scaffold_missing["cost_to_walmart_nat"].values
    )

scaffold = scaffold.drop(columns=["dsv_date"], errors="ignore")

pct_with_price = scaffold["cost_to_walmart"].notna().mean()
print(f"Rows with cost_to_walmart: {pct_with_price:.1%}")
print(f"Rows missing price: {scaffold['cost_to_walmart'].isna().sum():,}")

## 6. Walmart Offer Prices

In [ ]:
# Load item report (single snapshot for END_DATE)
# Note: assumes offer prices are relatively stable over the 30-day window
df_item_report = loader.load("dw_walmart_item_report", date_str=END_DATE)

# Keep relevant columns and filter to top SKUs
df_offer = df_item_report[["Product Code", "offer_price"]].copy()
df_offer = df_offer.rename(columns={"Product Code": "sku"})
df_offer = df_offer[df_offer["sku"].isin(top_skus)].drop_duplicates(subset="sku", keep="first")
df_offer["offer_price"] = pd.to_numeric(df_offer["offer_price"], errors="coerce")

print(f"Offer prices loaded: {len(df_offer):,} SKUs")

# Merge to scaffold
scaffold = scaffold.merge(df_offer, on="sku", how="left")
print(f"Rows with offer_price: {scaffold['offer_price'].notna().sum():,} / {len(scaffold):,}")

## 7. Supporting Data

In [ ]:
# MAP prices
df_map = loader.load("dw_map_prices", date_str=END_DATE)
df_map = df_map.rename(columns={"Product Code": "sku"})
df_map = df_map[["sku", "MAP"]].drop_duplicates(subset="sku", keep="first")
df_map["MAP"] = pd.to_numeric(df_map["MAP"], errors="coerce")
scaffold = scaffold.merge(df_map, on="sku", how="left")
print(f"SKUs with MAP: {scaffold['MAP'].notna().sum():,} rows")

# Shipping costs
df_shipping = loader.load("shipping_costs_by_node")
df_shipping = df_shipping.rename(columns={"Identifier": "node"})
df_shipping["node"] = df_shipping["node"].astype(str)
scaffold = scaffold.merge(df_shipping[["node", "Shipping cost"]], on="node", how="left")
scaffold = scaffold.rename(columns={"Shipping cost": "shipping_cost"})
print(f"Rows with shipping cost: {scaffold['shipping_cost'].notna().sum():,}")

In [ ]:
# Warehouse node mapping (for bridging Warehouse Code <-> Identifier)
df_wh_mapping = loader.load("warehouse_node_mapping")
df_wh_mapping["Identifier"] = df_wh_mapping["Identifier"].astype(str)
df_wh_mapping["Warehouse Code"] = df_wh_mapping["Warehouse Code"].astype(str)
node_to_wh = df_wh_mapping[["Identifier", "Warehouse Code"]].drop_duplicates(subset="Identifier")

# Warehouse addresses -> city/state
df_addresses = pd.read_csv(WAREHOUSE_ADDRESSES_PATH, dtype={"Code": str})
df_city = df_addresses[["Code", "Town", "State"]].rename(columns={"Code": "Warehouse Code"})

# Join: node -> Warehouse Code -> Town/State
node_city = node_to_wh.merge(df_city, on="Warehouse Code", how="left")
node_city = node_city.rename(columns={"Identifier": "node"})

scaffold = scaffold.merge(node_city[["node", "Town", "State"]], on="node", how="left")
print(f"Rows with city mapping: {scaffold['Town'].notna().sum():,} / {len(scaffold):,}")

## 8. Inventory (sampled every 4 days)

In [ ]:
# Lazy import pricing_module
ensure_modules_path()
import pricing_module as pricing

# Sample dates across the window
inv_dates = pd.date_range(start_dt, end_dt, freq=INVENTORY_SAMPLE_FREQ).strftime("%Y-%m-%d").tolist()
# Ensure END_DATE is included
if END_DATE not in inv_dates:
    inv_dates.append(END_DATE)

print(f"Inventory dates to load: {len(inv_dates)}")
print(inv_dates)

In [ ]:
# Load inventory for sampled dates
min_units = nlc_config["inventory"]["min_units_secondary"]  # 4

# Warehouse node mapping for filtering (WalmartB2B, ENABLED, Inventory Enabled=1)
df_wh_filt = df_wh_mapping[
    (df_wh_mapping["Channel"] == "WalmartB2B")
    & (df_wh_mapping["Warehouse Status"] == "ENABLED")
    & (df_wh_mapping["Identifier Status"] == "ENABLED")
    & (df_wh_mapping["Inventory Enabled"] == 1)
].copy()

inv_records = []
for date_i in inv_dates:
    logger.info("Loading inventory for %s...", date_i)
    df_inv_i = pricing.get_inventory(date_i, add_rebates=False, amazon=False, greater3=True)
    df_inv_i["date"] = pd.to_datetime(date_i)
    
    # Filter by min_units
    df_inv_i = df_inv_i[df_inv_i["Available"] >= min_units].copy()
    
    # Filter to top SKUs
    df_inv_i = df_inv_i[df_inv_i["Product Code"].isin(top_skus)].copy()
    
    # Merge with node mapping
    df_inv_i["Warehouse Code"] = df_inv_i["Warehouse Code"].astype(str)
    df_inv_i = df_inv_i.merge(
        df_wh_filt[["Warehouse Code", "Identifier", "Inventory Threshold"]],
        on="Warehouse Code", how="inner"
    )
    
    # Filter by Inventory Threshold
    df_inv_i = df_inv_i[df_inv_i["Available"] >= df_inv_i["Inventory Threshold"]].copy()
    
    # Per SKU-Node: min Purchase Price+FET
    df_inv_agg = (
        df_inv_i.groupby(["Product Code", "Identifier"])
        .agg({"Purchase Price+FET": "min"})
        .reset_index()
        .rename(columns={
            "Product Code": "sku",
            "Identifier": "node",
            "Purchase Price+FET": "min_purchase_price_fet",
        })
    )
    df_inv_agg["inv_date"] = pd.to_datetime(date_i)
    df_inv_agg["can_show_inv"] = 1
    inv_records.append(df_inv_agg)
    logger.info("  -> %d SKU-Node pairs with inventory", len(df_inv_agg))

df_inv_all = pd.concat(inv_records, ignore_index=True)
df_inv_all["node"] = df_inv_all["node"].astype(str)
print(f"Total inventory records: {len(df_inv_all):,}")


In [ ]:
# Forward-fill inventory to all analysis dates using merge_asof
scaffold = scaffold.sort_values("date")
df_inv_all = df_inv_all.sort_values("inv_date")

scaffold = pd.merge_asof(
    scaffold,
    df_inv_all[["sku", "node", "min_purchase_price_fet", "can_show_inv", "inv_date"]],
    left_on="date",
    right_on="inv_date",
    by=["sku", "node"],
    direction="backward",
)
scaffold["can_show_inv"] = scaffold["can_show_inv"].fillna(0).astype(int)
scaffold = scaffold.drop(columns=["inv_date"], errors="ignore")

print(f"Rows with inventory: {(scaffold['can_show_inv'] == 1).sum():,} / {len(scaffold):,}")
print(f"Rows with purchase price: {scaffold['min_purchase_price_fet'].notna().sum():,}")

## 9. Assemble Master DataFrame

In [ ]:
# Computed columns
scaffold["walmart_margin"] = (
    (scaffold["offer_price"] - scaffold["cost_to_walmart"]) / scaffold["offer_price"]
)
scaffold["te_margin"] = (
    (scaffold["cost_to_walmart"] - scaffold["min_purchase_price_fet"]) / scaffold["cost_to_walmart"]
)

# Brand = first 4 chars of SKU
scaffold["brand"] = scaffold["sku"].str[:4]

# Day of week
scaffold["day_of_week"] = scaffold["date"].dt.dayofweek  # 0=Mon, 6=Sun

# MAP proximity: how close cost_to_walmart is to the MAP floor
scaffold["map_proximity"] = scaffold["cost_to_walmart"] / (scaffold["MAP"] * 0.95)

# Number of active nodes per SKU per date
active_nodes = (
    scaffold[scaffold["can_show_inv"] == 1]
    .groupby(["sku", "date"])["node"]
    .nunique()
    .reset_index()
    .rename(columns={"node": "n_active_nodes"})
)
scaffold = scaffold.merge(active_nodes, on=["sku", "date"], how="left")
scaffold["n_active_nodes"] = scaffold["n_active_nodes"].fillna(0).astype(int)

print("Computed columns added.")
print(f"Columns: {scaffold.columns.tolist()}")

In [ ]:
# Days since last price change (from DSV history)
# Build a per-SKU-Node price change timeline from DSV snapshots
dsv_prices = (
    df_dsv_all[df_dsv_all["source"] != "nan"]
    .rename(columns={"source": "node_dsv"})
    .sort_values("dsv_date")
)

# Detect price changes per SKU-Node
dsv_prices["prev_price"] = dsv_prices.groupby(["sku", "node_dsv"])["cost_to_walmart"].shift(1)
dsv_prices["price_changed"] = (
    (dsv_prices["cost_to_walmart"] != dsv_prices["prev_price"])
    & dsv_prices["prev_price"].notna()
)

# Get change dates
change_dates = dsv_prices[dsv_prices["price_changed"]][["sku", "node_dsv", "dsv_date"]].copy()
change_dates = change_dates.rename(columns={"node_dsv": "node", "dsv_date": "change_date"})
change_dates = change_dates.sort_values("change_date")

# For each scaffold row, find most recent price change
if len(change_dates) > 0:
    scaffold = scaffold.sort_values("date")
    scaffold = pd.merge_asof(
        scaffold,
        change_dates,
        left_on="date",
        right_on="change_date",
        by=["sku", "node"],
        direction="backward",
    )
    scaffold["days_since_price_change"] = (scaffold["date"] - scaffold["change_date"]).dt.days
    scaffold = scaffold.drop(columns=["change_date"], errors="ignore")
else:
    scaffold["days_since_price_change"] = np.nan

print(f"Rows with days_since_price_change: {scaffold['days_since_price_change'].notna().sum():,}")

## 10. Rolling 7-Day Comparisons

In [ ]:
# Sort by SKU-Node-Date for rolling calculations
scaffold = scaffold.sort_values(["sku", "node", "date"]).reset_index(drop=True)

rolling_cols = ["qty_sold", "te_margin", "cost_to_walmart", "offer_price", "walmart_margin"]

for col in rolling_cols:
    # Rolling 7-day average of prior days (shift 1 to exclude current day)
    scaffold[f"{col}_7d_avg"] = (
        scaffold.groupby(["sku", "node"])[col]
        .transform(lambda x: x.shift(1).rolling(ROLLING_WINDOW, min_periods=1).mean())
    )
    # Delta and percentage change
    scaffold[f"{col}_vs_7d"] = scaffold[col] - scaffold[f"{col}_7d_avg"]
    scaffold[f"{col}_vs_7d_pct"] = (
        scaffold[f"{col}_vs_7d"] / scaffold[f"{col}_7d_avg"].replace(0, np.nan)
    )

print(f"Rolling comparison columns added for: {rolling_cols}")

In [ ]:
# Trim warmup period â€” keep only the analysis window
df = scaffold[scaffold["date"] >= start_dt].copy().reset_index(drop=True)

print(f"Final dataset shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"\nColumn types:")
print(df.dtypes)

## 11. Exploratory Data Analysis

In [ ]:
# Dataset summary
print(f"Shape: {df.shape}")
print(f"\nMissing values:")
missing = df.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))

print(f"\nDescriptive statistics:")
df.describe()

In [ ]:
# Missing values heatmap
fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(df.isnull().T, cbar=False, cmap="YlOrRd", ax=ax)
ax.set_title("Missing Values by Column")
plt.tight_layout()
plt.show()

In [ ]:
# Spearman correlation matrix (full numeric columns)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Remove ID-like columns
numeric_cols = [c for c in numeric_cols if c not in ["day_of_week"]]

corr_matrix = df[numeric_cols].corr(method="spearman")

fig, ax = plt.subplots(figsize=(18, 15))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, ax=ax,
    annot_kws={"size": 7}, square=True,
)
ax.set_title("Spearman Rank Correlation Matrix", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Top correlations with qty_sold
target_corrs = (
    corr_matrix["qty_sold"]
    .drop("qty_sold")
    .dropna()
    .sort_values(key=abs, ascending=False)
)

fig, ax = plt.subplots(figsize=(12, 8))
colors = ["#e74c3c" if v < 0 else "#2ecc71" for v in target_corrs.head(15)]
target_corrs.head(15).plot.barh(ax=ax, color=colors)
ax.set_xlabel("Spearman Correlation with qty_sold")
ax.set_title("Top 15 Features Correlated with Sales Volume")
ax.axvline(x=0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

print("\nTop 15 correlations with qty_sold:")
print(target_corrs.head(15).to_string())

In [ ]:
# Scatter plots for top 6 correlated features (individual figures for clarity)
top_features = target_corrs.head(6).index.tolist()

# Sample for performance
df_nonzero = df[df["qty_sold"] > 0]
df_sample = df_nonzero.sample(n=min(10000, len(df_nonzero)), random_state=42)

for col in top_features:
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.regplot(
        data=df_sample, x=col, y="qty_sold", ax=ax,
        scatter_kws={"alpha": 0.2, "s": 8}, line_kws={"color": "red"},
    )
    valid = df_sample[[col, "qty_sold"]].dropna()
    r, p = stats.spearmanr(valid[col], valid["qty_sold"])
    ax.set_title(f"{col} vs qty_sold  (Spearman r={r:.3f}, p={p:.2e})", fontsize=13)
    ax.set_xlabel(col, fontsize=11)
    ax.set_ylabel("qty_sold", fontsize=11)
    plt.tight_layout()
    plt.show()


In [ ]:
# Distribution plots for key variables (individual figures)
dist_cols = ["qty_sold", "walmart_margin", "te_margin", "cost_to_walmart", "days_since_price_change"]
dist_cols = [c for c in dist_cols if c in df.columns]

for col in dist_cols:
    fig, ax = plt.subplots(figsize=(10, 5))
    data = df[col].dropna()
    if col == "qty_sold":
        data = data[data > 0]  # Skip zeros for visibility
        ax.set_title(f"Distribution of {col} (non-zero only)", fontsize=13)
    else:
        ax.set_title(f"Distribution of {col}", fontsize=13)
    sns.histplot(data, ax=ax, bins=50, kde=True)
    ax.set_xlabel(col, fontsize=11)
    plt.tight_layout()
    plt.show()


## 12. Statistical Tests

In [ ]:
# Test 1: Price change impact on sales
# Compare qty_sold on days with a recent price change (<=3 days) vs no change (>7 days)
recent_change = df[df["days_since_price_change"] <= 3]["qty_sold"]
no_change = df[df["days_since_price_change"] > 7]["qty_sold"]

if len(recent_change) > 0 and len(no_change) > 0:
    stat, pval = stats.mannwhitneyu(recent_change, no_change, alternative="two-sided")
    print("=== Price Change Impact on Sales ===")
    print(f"Recent change (<=3 days): n={len(recent_change):,}, mean={recent_change.mean():.3f}, median={recent_change.median():.1f}")
    print(f"No recent change (>7 days): n={len(no_change):,}, mean={no_change.mean():.3f}, median={no_change.median():.1f}")
    print(f"Mann-Whitney U stat={stat:.0f}, p-value={pval:.4e}")
    print(f"Significant at 5%: {'Yes' if pval < 0.05 else 'No'}")
else:
    print("Insufficient data for price change analysis")

In [ ]:
# Test 2: Margin level vs. revenue
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, col, label in zip(axes, ["te_margin", "walmart_margin"], ["TE Margin", "Walmart Margin"]):
    df_valid = df[df[col].notna() & (df["qty_sold"] > 0)].copy()
    if len(df_valid) > 100:
        df_valid["margin_bin"] = pd.qcut(df_valid[col], 10, duplicates="drop")
        margin_revenue = (
            df_valid.groupby("margin_bin", observed=True)
            .agg(avg_qty=("qty_sold", "mean"), total_qty=("qty_sold", "sum"), count=("qty_sold", "count"))
            .reset_index()
        )
        margin_revenue["margin_bin"] = margin_revenue["margin_bin"].astype(str)
        ax.bar(range(len(margin_revenue)), margin_revenue["avg_qty"])
        ax.set_xticks(range(len(margin_revenue)))
        ax.set_xticklabels(margin_revenue["margin_bin"], rotation=45, ha="right", fontsize=8)
        ax.set_ylabel("Avg Qty Sold")
        ax.set_title(f"Avg Sales by {label} Decile")
    else:
        ax.text(0.5, 0.5, "Insufficient data", ha="center", va="center", transform=ax.transAxes)

plt.tight_layout()
plt.show()

In [ ]:
# Test 3: Inventory availability vs sales
inv_yes = df[df["can_show_inv"] == 1]["qty_sold"]
inv_no = df[df["can_show_inv"] == 0]["qty_sold"]

print("=== Inventory Availability vs Sales ===")
print(f"With inventory:    n={len(inv_yes):,}, mean={inv_yes.mean():.3f}, median={inv_yes.median():.1f}")
print(f"Without inventory: n={len(inv_no):,}, mean={inv_no.mean():.3f}, median={inv_no.median():.1f}")

if len(inv_yes) > 0 and len(inv_no) > 0:
    stat, pval = stats.mannwhitneyu(inv_yes, inv_no, alternative="two-sided")
    print(f"Mann-Whitney U stat={stat:.0f}, p-value={pval:.4e}")
    print(f"Significant at 5%: {'Yes' if pval < 0.05 else 'No'}")

In [ ]:
# Test 4: OLS regression
import statsmodels.api as sm

feature_cols = [
    "cost_to_walmart", "offer_price", "walmart_margin", "te_margin",
    "can_show_inv", "shipping_cost", "n_active_nodes", "day_of_week",
    "days_since_price_change",
]
feature_cols = [c for c in feature_cols if c in df.columns]

df_reg = df[feature_cols + ["qty_sold"]].dropna()

if len(df_reg) > 100:
    X = sm.add_constant(df_reg[feature_cols])
    y = df_reg["qty_sold"]
    model = sm.OLS(y, X).fit()
    print(model.summary())
else:
    print("Insufficient data for OLS regression")

In [ ]:
# Also run with log-transformed qty_sold (to handle zero-inflation)
df_reg_log = df_reg[df_reg["qty_sold"] > 0].copy()
if len(df_reg_log) > 100:
    X_log = sm.add_constant(df_reg_log[feature_cols])
    y_log = np.log1p(df_reg_log["qty_sold"])
    model_log = sm.OLS(y_log, X_log).fit()
    print("=== OLS with log(1 + qty_sold), non-zero sales only ===")
    print(model_log.summary())
else:
    print("Insufficient non-zero sales data for log OLS")

### Test 5: Price Change Impact on Revenue

Study the correlation between Walmart price changes and revenue directly,
complementing the qty-based analysis in Test 1.

In [ ]:
# Price Change Impact on Revenue
if "revenue" in df.columns and "cost_to_walmart_pct_chg" in df.columns:
    df_pc = df[df["cost_to_walmart_pct_chg"].notna() & df["revenue"].notna()].copy()
    df_pc = df_pc[df_pc["cost_to_walmart_pct_chg"] != 0]

    if len(df_pc) > 50:
        from scipy.stats import spearmanr, pearsonr, mannwhitneyu

        r_pearson, p_pearson = pearsonr(df_pc["cost_to_walmart_pct_chg"], df_pc["revenue"])
        r_spearman, p_spearman = spearmanr(df_pc["cost_to_walmart_pct_chg"], df_pc["revenue"])
        print("=== Price Change vs Revenue Correlation ===")
        print(f"Pearson  r = {r_pearson:.4f}  (p = {p_pearson:.4g})")
        print(f"Spearman r = {r_spearman:.4f}  (p = {p_spearman:.4g})")

        # Bucketed analysis
        df_pc["price_change_bucket"] = pd.cut(
            df_pc["cost_to_walmart_pct_chg"],
            bins=[-np.inf, -0.05, -0.01, 0.01, 0.05, np.inf],
            labels=["Large decrease (<-5%)", "Small decrease (-5% to -1%)",
                    "Minimal (-1% to 1%)", "Small increase (1% to 5%)",
                    "Large increase (>5%)"],
        )
        bucket_stats = (
            df_pc.groupby("price_change_bucket", observed=True)
            .agg(n=("revenue", "size"), mean_revenue=("revenue", "mean"),
                 median_revenue=("revenue", "median"), total_revenue=("revenue", "sum"),
                 mean_qty=("qty_sold", "mean"))
            .round(2)
        )
        print()
        print("=== Revenue by Price Change Bucket ===")
        print(bucket_stats.to_string())

        # Visualization
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        axes[0].scatter(df_pc["cost_to_walmart_pct_chg"] * 100, df_pc["revenue"], alpha=0.15, s=8)
        axes[0].set_xlabel("Price Change (%)")
        axes[0].set_ylabel("Revenue ($)")
        axes[0].set_title(f"Price Change vs Revenue (r={r_spearman:.3f})")
        axes[0].axvline(0, color="red", linestyle="--", alpha=0.5)

        bucket_stats["mean_revenue"].plot(kind="bar", ax=axes[1], color="steelblue", alpha=0.8)
        axes[1].set_ylabel("Mean Revenue ($)")
        axes[1].set_title("Mean Revenue by Price Change Bucket")
        axes[1].tick_params(axis="x", rotation=30)

        bucket_stats["mean_qty"].plot(kind="bar", ax=axes[2], color="darkorange", alpha=0.8)
        axes[2].set_ylabel("Mean Qty Sold")
        axes[2].set_title("Mean Qty by Price Change Bucket")
        axes[2].tick_params(axis="x", rotation=30)
        plt.tight_layout()
        plt.show()

        # Pre/post revenue around price change events
        print()
        print("=== Pre/Post Revenue Around Price Change Events ===")
        df_events = df[df["days_since_price_change"].notna()].copy()
        pre_rev = df_events[df_events["days_since_price_change"].between(4, 7)]["revenue"]
        post_rev = df_events[df_events["days_since_price_change"].between(0, 3)]["revenue"]
        print(f"Pre-change (4-7 days before):  n={len(pre_rev):,}, mean=${pre_rev.mean():.2f}")
        print(f"Post-change (0-3 days after):  n={len(post_rev):,}, mean=${post_rev.mean():.2f}")
        if len(pre_rev) > 30 and len(post_rev) > 30:
            stat, pval = mannwhitneyu(post_rev, pre_rev, alternative="two-sided")
            print(f"Mann-Whitney U test: stat={stat:.1f}, p={pval:.4g}")
            pct_change = (post_rev.mean() - pre_rev.mean()) / pre_rev.mean() * 100
            print(f"Revenue change: {pct_change:+.1f}%")
    else:
        print(f"Insufficient price-change rows ({len(df_pc)}) for revenue correlation analysis")
else:
    missing = [c for c in ["revenue", "cost_to_walmart_pct_chg"] if c not in df.columns]
    print(f"Columns missing for price-revenue analysis: {missing}")


## 13. Geographic & Brand Analysis

In [ ]:
# Sales by state
state_sales = (
    df.groupby("State")
    .agg(
        total_qty=("qty_sold", "sum"),
        avg_wm_margin=("walmart_margin", "mean"),
        avg_te_margin=("te_margin", "mean"),
        n_nodes=("node", "nunique"),
    )
    .sort_values("total_qty", ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(14, 6))
state_sales["total_qty"].plot.bar(ax=ax)
ax.set_title("Total Sales by State (Top 20)")
ax.set_ylabel("Total Qty Sold")
ax.set_xlabel("State")
plt.tight_layout()
plt.show()

state_sales

In [ ]:
# Brand-level analysis
brand_analysis = (
    df.groupby("brand")
    .agg(
        total_qty=("qty_sold", "sum"),
        total_revenue=("revenue", "sum"),
        avg_te_margin=("te_margin", "mean"),
        avg_wm_margin=("walmart_margin", "mean"),
        n_skus=("sku", "nunique"),
        n_nodes=("node", "nunique"),
    )
    .sort_values("total_qty", ascending=False)
    .head(20)
)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

brand_analysis["total_qty"].plot.bar(ax=axes[0])
axes[0].set_title("Total Sales by Brand (Top 20)")
axes[0].set_ylabel("Total Qty Sold")

brand_analysis[["avg_te_margin", "avg_wm_margin"]].plot.bar(ax=axes[1])
axes[1].set_title("Average Margins by Brand (Top 20)")
axes[1].set_ylabel("Margin")
axes[1].legend(["TE Margin", "Walmart Margin"])

plt.tight_layout()
plt.show()

brand_analysis

In [ ]:
# Node distribution breadth vs sales
breadth = (
    df.groupby("sku")
    .agg(
        avg_active_nodes=("n_active_nodes", "mean"),
        total_qty=("qty_sold", "sum"),
    )
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=breadth, x="avg_active_nodes", y="total_qty", alpha=0.4, s=15, ax=ax)
ax.set_xlabel("Avg Active Nodes per SKU")
ax.set_ylabel("Total Qty Sold (30 days)")
ax.set_title("Distribution Breadth vs. Sales Volume")

r, p = stats.spearmanr(breadth["avg_active_nodes"], breadth["total_qty"])
ax.text(0.05, 0.95, f"Spearman r={r:.3f}, p={p:.2e}", transform=ax.transAxes, va="top")

plt.tight_layout()
plt.show()

## 14. Summary & Cleanup

### Key Findings

*(Fill in after running the analysis)*

1. **Price change impact:** ...
2. **Top correlated features:** ...
3. **Margin sweet spots:** ...
4. **Inventory availability effect:** ...
5. **Geographic patterns:** ...
6. **Brand insights:** ...
7. **Distribution breadth:** ...

## 15. Extended Feature Engineering

Add tire size (from item report) and MAP tire flag to the dataset.

In [ ]:
# --- Discover tire-size column from the item report table ---
# Run a 1-row query to inspect all available columns
SIZE_COLUMN = None  # Will be set after inspection

try:
    schema_df = loader.dw.run_query(
        "SELECT * FROM pricing_tests.walmart_item_report",
        new_credentials=True,
    )
    print("Columns in pricing_tests.walmart_item_report:")
    for c in schema_df.columns:
        print(f"  {c:40s}  dtype={schema_df[c].dtype}  sample={schema_df[c].iloc[0]}")
except Exception as e:
    print(f"Schema inspection failed: {e}")
    print("Set SIZE_COLUMN manually below and re-run the next cell.")


In [ ]:
# --- Load tire size & create is_MAP_tire flag ---
import re as _re

# Set SIZE_COLUMN after inspecting the schema above
# SIZE_COLUMN = "product_name"  # <-- adjust to the correct column name

if SIZE_COLUMN is not None:
    size_query = f"""
        SELECT product_code AS "Product Code", {SIZE_COLUMN} AS raw_size
        FROM pricing_tests.walmart_item_report
        WHERE date = (SELECT MAX(date) FROM pricing_tests.walmart_item_report
                      WHERE date <= '{END_DATE}')
          AND "1p_offer_status" = 'PUBLISHED'
    """
    try:
        df_size = loader.dw.run_query(size_query, new_credentials=True)
        df_size = df_size.drop_duplicates(subset=["Product Code"])

        # Extract tire diameter from patterns like 205/55R16, LT265/70R17, 16", etc.
        df_size["tire_diameter"] = (
            df_size["raw_size"]
            .astype(str)
            .str.extract(r"[Rr](\d{2,3})")[0]
            .astype(float)
        )
        df_size["tire_size"] = df_size["raw_size"]

        df = df.merge(
            df_size[["Product Code", "tire_size", "tire_diameter"]],
            left_on="sku", right_on="Product Code", how="left",
        )
        if "Product Code_y" in df.columns:
            df.drop(columns=[c for c in df.columns if c.endswith("_y")], inplace=True)
            df.rename(columns={c: c.replace("_x", "") for c in df.columns if c.endswith("_x")}, inplace=True)

        print(f"Tire size coverage: {df['tire_size'].notna().mean():.1%}")
        print(f"Tire diameter distribution:\n{df['tire_diameter'].value_counts().head(10)}")
    except Exception as e:
        print(f"Tire size load failed: {e}")
        df["tire_size"] = np.nan
        df["tire_diameter"] = np.nan
else:
    print("SIZE_COLUMN not set — skipping tire size. Set it above and re-run.")
    df["tire_size"] = np.nan
    df["tire_diameter"] = np.nan

# MAP tire flag: True if the SKU has a MAP price (minimum allowed price)
df["is_MAP_tire"] = df["MAP"].notna()
print(f"\nMAP tire distribution:\n{df['is_MAP_tire'].value_counts()}")
print(f"MAP tire %: {df['is_MAP_tire'].mean():.1%}")


## 16. Price Elasticity by State & Brand-State

Estimate price elasticity at the state and brand-state level using log-log OLS:
`log(qty+1) ~ log(offer_price)`

Elasticity < 0 means higher price → lower sales (expected). More negative = more price-sensitive.

In [ ]:
# --- Elasticity by State ---
import statsmodels.api as sm

df_elast = df[(df["qty_sold"] > 0) & (df["offer_price"] > 0)].copy()

MIN_OBS_STATE = 50
state_elasticity = []

for state, grp in df_elast.groupby("State"):
    if pd.isna(state) or len(grp) < MIN_OBS_STATE:
        continue
    try:
        X = sm.add_constant(np.log(grp["offer_price"]))
        y = np.log1p(grp["qty_sold"])
        model = sm.OLS(y, X).fit()
        coef = model.params.iloc[-1]
        se = model.bse.iloc[-1]
        pval = model.pvalues.iloc[-1]
        state_elasticity.append({
            "State": state, "elasticity": coef, "SE": se, "p_value": pval,
            "n_obs": len(grp), "ci_lower": coef - 1.96 * se, "ci_upper": coef + 1.96 * se,
        })
    except Exception:
        continue

df_state_elast = pd.DataFrame(state_elasticity).sort_values("elasticity")
print(f"States with valid elasticity estimates: {len(df_state_elast)}")

# Bar chart: top/bottom 10
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_elastic = df_state_elast.head(10)
axes[0].barh(top_elastic["State"], top_elastic["elasticity"], color="#e74c3c", alpha=0.7)
axes[0].set_title("Top 10 Most Price-Sensitive States")
axes[0].set_xlabel("Elasticity (more negative = more sensitive)")

top_inelastic = df_state_elast.tail(10)
axes[1].barh(top_inelastic["State"], top_inelastic["elasticity"], color="#2ecc71", alpha=0.7)
axes[1].set_title("Top 10 Least Price-Sensitive States")
axes[1].set_xlabel("Elasticity")

plt.tight_layout()
plt.show()

display(df_state_elast.style.format({"elasticity": "{:.4f}", "SE": "{:.4f}", "p_value": "{:.4f}",
                                      "ci_lower": "{:.4f}", "ci_upper": "{:.4f}"}))


In [ ]:
# --- Brand-State Cross-Tabulation ---
MIN_OBS_BRAND_STATE = 30

top_brands_elast = df_elast.groupby("brand")["qty_sold"].sum().nlargest(15).index.tolist()
top_states_elast = df_elast.groupby("State")["qty_sold"].sum().nlargest(15).index.tolist()

brand_state_results = []
for brand in top_brands_elast:
    for state in top_states_elast:
        subset = df_elast[(df_elast["brand"] == brand) & (df_elast["State"] == state)]
        if len(subset) < MIN_OBS_BRAND_STATE:
            brand_state_results.append({"brand": brand, "State": state, "elasticity": np.nan,
                                         "n": len(subset), "p_value": np.nan})
            continue
        try:
            X = sm.add_constant(np.log(subset["offer_price"]))
            y = np.log1p(subset["qty_sold"])
            model = sm.OLS(y, X).fit()
            brand_state_results.append({
                "brand": brand, "State": state,
                "elasticity": model.params.iloc[-1],
                "n": len(subset), "p_value": model.pvalues.iloc[-1],
            })
        except Exception:
            brand_state_results.append({"brand": brand, "State": state, "elasticity": np.nan,
                                         "n": len(subset), "p_value": np.nan})

df_bs_elast = pd.DataFrame(brand_state_results)

# Pivot heatmap
pivot = df_bs_elast.pivot(index="brand", columns="State", values="elasticity")
fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(pivot, cmap="RdYlGn", center=0, annot=True, fmt=".2f",
            linewidths=0.5, ax=ax, cbar_kws={"label": "Elasticity"})
ax.set_title("Brand-State Price Elasticity (negative = price-sensitive)")
plt.tight_layout()
plt.show()

# Narrative: top 5 most and least elastic segments
df_bs_valid = df_bs_elast.dropna(subset=["elasticity"])
most_elastic = df_bs_valid.nsmallest(5, "elasticity")
least_elastic = df_bs_valid.nlargest(5, "elasticity")

print("\n=== TOP 5 MOST PRICE-SENSITIVE Brand-State Segments ===")
for _, r in most_elastic.iterrows():
    sig = "***" if r["p_value"] < 0.001 else "**" if r["p_value"] < 0.01 else "*" if r["p_value"] < 0.05 else ""
    print(f"  {r['brand']} in {r['State']}: elasticity = {r['elasticity']:.3f}{sig}  (n={r['n']:,})")

print("\n=== TOP 5 LEAST PRICE-SENSITIVE Brand-State Segments ===")
for _, r in least_elastic.iterrows():
    sig = "***" if r["p_value"] < 0.001 else "**" if r["p_value"] < 0.01 else "*" if r["p_value"] < 0.05 else ""
    print(f"  {r['brand']} in {r['State']}: elasticity = {r['elasticity']:.3f}{sig}  (n={r['n']:,})")


In [ ]:
# --- City-Level Drill-Down ---
MIN_OBS_CITY = 30
MIN_BRAND_STATE_FOR_CITY = 200

city_results = []
for brand in top_brands_elast:
    for state in top_states_elast:
        bs_subset = df_elast[(df_elast["brand"] == brand) & (df_elast["State"] == state)]
        if len(bs_subset) < MIN_BRAND_STATE_FOR_CITY:
            continue
        for city, city_grp in bs_subset.groupby("Town"):
            if pd.isna(city) or len(city_grp) < MIN_OBS_CITY:
                continue
            try:
                X = sm.add_constant(np.log(city_grp["offer_price"]))
                y = np.log1p(city_grp["qty_sold"])
                model = sm.OLS(y, X).fit()
                city_results.append({
                    "brand": brand, "State": state, "Town": city,
                    "elasticity": model.params.iloc[-1],
                    "n": len(city_grp), "p_value": model.pvalues.iloc[-1],
                })
            except Exception:
                continue

if city_results:
    df_city_elast = pd.DataFrame(city_results)
    # Show only brand-state combos with meaningful city-level variation
    city_var = df_city_elast.groupby(["brand", "State"])["elasticity"].std().reset_index()
    city_var = city_var.rename(columns={"elasticity": "elasticity_std"})
    notable = city_var[city_var["elasticity_std"] > 0.5]

    if len(notable) > 0:
        print(f"Brand-State segments with high city-level elasticity variation (std > 0.5): {len(notable)}")
        for _, r in notable.iterrows():
            cities = df_city_elast[(df_city_elast["brand"] == r["brand"]) & (df_city_elast["State"] == r["State"])]
            print(f"\n  {r['brand']} in {r['State']} (std={r['elasticity_std']:.2f}):")
            for _, c in cities.sort_values("elasticity").iterrows():
                sig = "*" if c["p_value"] < 0.05 else ""
                print(f"    {c['Town']:25s} elasticity={c['elasticity']:.3f}{sig} (n={c['n']:,})")
    else:
        print("No brand-state segments show meaningful city-level variation (std > 0.5).")
        print("City-level sample sizes may be too small for reliable estimates.")
else:
    print("Insufficient data for city-level elasticity estimates (need n>=30 per city).")


## 17. Heterogeneous Treatment Effects (DiD Decomposition)

Break down the Difference-in-Differences causal estimate by brand, price tier, state,
and MAP status to identify which segments are most/least affected by price changes.

In [ ]:
# --- Build DiD Panel ---
# Identify treated SKU-Nodes (those with a price change) and matched controls

df_did = df[df["days_since_price_change"].notna()].copy()

# SKU_Node key
if "SKU_Node" not in df_did.columns:
    df_did["SKU_Node"] = df_did["sku"] + "-" + df_did["node"].astype(str)
if "SKU_Node" not in df.columns:
    df["SKU_Node"] = df["sku"] + "-" + df["node"].astype(str)

# Treated: nodes with non-zero cost change
_changed = df_did["cost_to_walmart_vs_7d"].fillna(0) != 0
treated_candidates = set(df_did.loc[_changed, "SKU_Node"].unique())
all_nodes = set(df_did["SKU_Node"].unique())
control_nodes = all_nodes - treated_candidates

# First event date per treated node
price_events = (
    df_did.loc[_changed]
    .groupby("SKU_Node", sort=False)["date"].min()
    .reset_index().rename(columns={"date": "event_date"})
)

# Require 14-day pre/post window
min_event = pd.Timestamp(START_DATE) + pd.Timedelta(days=14)
max_event = pd.Timestamp(END_DATE) - pd.Timedelta(days=14)
price_events = price_events[
    (price_events["event_date"] >= min_event) & (price_events["event_date"] <= max_event)
]
treated_nodes = set(price_events["SKU_Node"])

# Brand lookup
treated_brands = df[df["SKU_Node"].isin(treated_nodes)].drop_duplicates("SKU_Node").set_index("SKU_Node")["brand"]
control_brand_map = df[df["SKU_Node"].isin(control_nodes)].drop_duplicates("SKU_Node").set_index("SKU_Node")["brand"]

# Match up to 5 controls per brand
brand_to_controls = control_brand_map.groupby(control_brand_map).apply(
    lambda x: x.index[:5].tolist()
).to_dict()

pe = price_events.copy()
pe["brand"] = pe["SKU_Node"].map(treated_brands)
pe = pe.dropna(subset=["brand"])

ctrl_rows = []
for _, row in pe.iterrows():
    for c in brand_to_controls.get(row["brand"], []):
        ctrl_rows.append({"SKU_Node": c, "event_date": row["event_date"]})
df_ctrl_assign = pd.DataFrame(ctrl_rows) if ctrl_rows else pd.DataFrame(columns=["SKU_Node", "event_date"])

df_treat_assign = pe[["SKU_Node", "event_date"]].copy()
df_treat_assign["treated"] = 1
df_ctrl_assign["treated"] = 0
all_assign = pd.concat([df_treat_assign, df_ctrl_assign], ignore_index=True)

# Merge with df
panel_cols = ["SKU_Node", "date", "qty_sold", "revenue", "brand", "State",
              "cost_to_walmart", "offer_price", "te_margin", "walmart_margin",
              "day_of_week", "can_show_inv", "n_active_nodes", "is_MAP_tire"]
panel_cols = [c for c in panel_cols if c in df.columns]
df_did_panel = all_assign.merge(df[panel_cols], on="SKU_Node", how="inner")

# Filter to 7-day pre / 14-day post window
df_did_panel["_delta"] = (df_did_panel["date"] - df_did_panel["event_date"]).dt.days
df_did_panel = df_did_panel[
    (df_did_panel["_delta"] >= -7) & (df_did_panel["_delta"] <= 14)
].copy()
df_did_panel["post"] = (df_did_panel["date"] >= df_did_panel["event_date"]).astype(int)
df_did_panel["treated_x_post"] = df_did_panel["treated"] * df_did_panel["post"]
df_did_panel.drop(columns=["_delta"], inplace=True)

# Price tier (quartile of offer_price within brand)
def _safe_qcut(x):
    try:
        return pd.qcut(x, 4, labels=["Budget", "Mid-Low", "Mid-High", "Premium"], duplicates="drop")
    except ValueError:
        try:
            return pd.qcut(x, 2, labels=["Budget", "Premium"], duplicates="drop")
        except ValueError:
            return pd.Series(["Budget"] * len(x), index=x.index)

df_did_panel["price_tier"] = df_did_panel.groupby("brand")["offer_price"].transform(_safe_qcut)

print(f"DiD panel: {len(df_did_panel):,} rows")
print(f"Treated: {(df_did_panel['treated']==1).sum():,}, Control: {(df_did_panel['treated']==0).sum():,}")


In [ ]:
# --- Heterogeneous DiD Regressions ---
het_results = []

if len(df_did_panel) > 100:
    # 1. By Brand (top 10)
    top_did_brands = df_did_panel["brand"].value_counts().head(10).index.tolist()
    for brand in top_did_brands:
        sub = df_did_panel[df_did_panel["brand"] == brand]
        if len(sub) < 50:
            continue
        sub_reg = sub[["qty_sold", "treated", "post", "treated_x_post", "day_of_week"]].dropna()
        if len(sub_reg) < 50:
            continue
        X = sm.add_constant(sub_reg[["treated", "post", "treated_x_post", "day_of_week"]])
        y = sub_reg["qty_sold"]
        try:
            model = sm.OLS(y, X).fit(cov_type="HC1")
            het_results.append({
                "segment_type": "Brand", "segment_value": brand,
                "ATT": model.params["treated_x_post"], "SE": model.bse["treated_x_post"],
                "p_value": model.pvalues["treated_x_post"], "n_obs": len(sub_reg),
            })
        except Exception:
            continue

    # 2. By Price Tier
    for tier in ["Budget", "Mid-Low", "Mid-High", "Premium"]:
        sub = df_did_panel[df_did_panel["price_tier"] == tier]
        if len(sub) < 50:
            continue
        sub_reg = sub[["qty_sold", "treated", "post", "treated_x_post", "day_of_week"]].dropna()
        if len(sub_reg) < 50:
            continue
        X = sm.add_constant(sub_reg[["treated", "post", "treated_x_post", "day_of_week"]])
        y = sub_reg["qty_sold"]
        try:
            model = sm.OLS(y, X).fit(cov_type="HC1")
            het_results.append({
                "segment_type": "Price Tier", "segment_value": tier,
                "ATT": model.params["treated_x_post"], "SE": model.bse["treated_x_post"],
                "p_value": model.pvalues["treated_x_post"], "n_obs": len(sub_reg),
            })
        except Exception:
            continue

    # 3. By State (top 10)
    top_did_states = df_did_panel["State"].value_counts().head(10).index.tolist()
    for state in top_did_states:
        if pd.isna(state):
            continue
        sub = df_did_panel[df_did_panel["State"] == state]
        if len(sub) < 50:
            continue
        sub_reg = sub[["qty_sold", "treated", "post", "treated_x_post", "day_of_week"]].dropna()
        if len(sub_reg) < 50:
            continue
        X = sm.add_constant(sub_reg[["treated", "post", "treated_x_post", "day_of_week"]])
        y = sub_reg["qty_sold"]
        try:
            model = sm.OLS(y, X).fit(cov_type="HC1")
            het_results.append({
                "segment_type": "State", "segment_value": state,
                "ATT": model.params["treated_x_post"], "SE": model.bse["treated_x_post"],
                "p_value": model.pvalues["treated_x_post"], "n_obs": len(sub_reg),
            })
        except Exception:
            continue

    # 4. By MAP status
    for is_map in [True, False]:
        sub = df_did_panel[df_did_panel["is_MAP_tire"] == is_map]
        if len(sub) < 50:
            continue
        sub_reg = sub[["qty_sold", "treated", "post", "treated_x_post", "day_of_week"]].dropna()
        if len(sub_reg) < 50:
            continue
        X = sm.add_constant(sub_reg[["treated", "post", "treated_x_post", "day_of_week"]])
        y = sub_reg["qty_sold"]
        try:
            model = sm.OLS(y, X).fit(cov_type="HC1")
            het_results.append({
                "segment_type": "MAP Status", "segment_value": "MAP Tire" if is_map else "Non-MAP",
                "ATT": model.params["treated_x_post"], "SE": model.bse["treated_x_post"],
                "p_value": model.pvalues["treated_x_post"], "n_obs": len(sub_reg),
            })
        except Exception:
            continue

df_het = pd.DataFrame(het_results)
if len(df_het) > 0:
    df_het["sig"] = df_het["p_value"].apply(
        lambda p: "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
    )
    print("\n=== Heterogeneous Treatment Effects (ATT = avg daily qty change from price change) ===\n")
    for seg_type in ["Brand", "Price Tier", "State", "MAP Status"]:
        seg_data = df_het[df_het["segment_type"] == seg_type]
        if len(seg_data) == 0:
            continue
        print(f"\n--- By {seg_type} ---")
        for _, r in seg_data.sort_values("ATT").iterrows():
            print(f"  {r['segment_value']:20s}  ATT={r['ATT']:+.4f}{r['sig']}  SE={r['SE']:.4f}  n={r['n_obs']:,}")
    display(df_het.style.format({"ATT": "{:+.4f}", "SE": "{:.4f}", "p_value": "{:.4f}"}))
else:
    print("Insufficient data for heterogeneous treatment effects.")


## 18. Seasonal Price Sensitivity

Does price elasticity vary across sub-periods within the 90-day analysis window?
Split into 3 monthly sub-periods and compare elasticity estimates.

In [ ]:
# --- Sub-period elasticity estimation ---
# Split the 90-day window into 3 ~30-day periods
date_range = pd.date_range(start=START_DATE, end=END_DATE, periods=4)
period_labels = [f"P1 ({date_range[0].strftime('%b %d')}-{date_range[1].strftime('%b %d')})",
                 f"P2 ({date_range[1].strftime('%b %d')}-{date_range[2].strftime('%b %d')})",
                 f"P3 ({date_range[2].strftime('%b %d')}-{date_range[3].strftime('%b %d')})"]

df["sub_period"] = pd.cut(df["date"], bins=date_range, labels=period_labels, include_lowest=True)

df_elast_s = df[(df["qty_sold"] > 0) & (df["offer_price"] > 0)].copy()
top_brands_s = df_elast_s.groupby("brand")["qty_sold"].sum().nlargest(15).index.tolist()

seasonal_results = []
for period in period_labels:
    p_data = df_elast_s[df_elast_s["sub_period"] == period]
    # Overall elasticity for the period
    if len(p_data) >= 50:
        X = sm.add_constant(np.log(p_data["offer_price"]))
        y = np.log1p(p_data["qty_sold"])
        try:
            model = sm.OLS(y, X).fit()
            seasonal_results.append({"period": period, "brand": "ALL", "elasticity": model.params.iloc[-1],
                                      "SE": model.bse.iloc[-1], "p_value": model.pvalues.iloc[-1], "n": len(p_data)})
        except Exception:
            pass
    # Per-brand elasticity
    for brand in top_brands_s:
        b_data = p_data[p_data["brand"] == brand]
        if len(b_data) < 30:
            continue
        try:
            X = sm.add_constant(np.log(b_data["offer_price"]))
            y = np.log1p(b_data["qty_sold"])
            model = sm.OLS(y, X).fit()
            seasonal_results.append({"period": period, "brand": brand, "elasticity": model.params.iloc[-1],
                                      "SE": model.bse.iloc[-1], "p_value": model.pvalues.iloc[-1], "n": len(b_data)})
        except Exception:
            continue

df_seasonal = pd.DataFrame(seasonal_results)

# Test for significant seasonal variation: interaction model
print("=== Overall Elasticity by Sub-Period ===")
overall = df_seasonal[df_seasonal["brand"] == "ALL"]
display(overall[["period", "elasticity", "SE", "p_value", "n"]].style.format(
    {"elasticity": "{:.4f}", "SE": "{:.4f}", "p_value": "{:.4f}"}))

# Interaction test: log(qty+1) ~ log(price) * period_dummy
if len(df_elast_s) > 100 and df_elast_s["sub_period"].nunique() > 1:
    df_interact = df_elast_s.copy()
    period_dummies = pd.get_dummies(df_interact["sub_period"], prefix="period", drop_first=True, dtype=float)
    df_interact = pd.concat([df_interact, period_dummies], axis=1)
    log_price = np.log(df_interact["offer_price"])
    interact_cols = []
    for col in period_dummies.columns:
        interact_col = f"logp_x_{col}"
        df_interact[interact_col] = log_price * df_interact[col]
        interact_cols.append(interact_col)
    X = sm.add_constant(df_interact[["offer_price"] + list(period_dummies.columns) + interact_cols].apply(pd.to_numeric, errors="coerce").dropna(axis=1))
    X = sm.add_constant(pd.concat([log_price, period_dummies, df_interact[interact_cols]], axis=1).dropna())
    y = np.log1p(df_interact["qty_sold"]).loc[X.index]
    try:
        model = sm.OLS(y, X).fit(cov_type="HC1")
        print("\n=== Interaction Model (seasonal variation in elasticity) ===")
        for col in interact_cols:
            if col in model.params.index:
                sig = "***" if model.pvalues[col] < 0.001 else "**" if model.pvalues[col] < 0.01 else "*" if model.pvalues[col] < 0.05 else ""
                print(f"  {col}: coef={model.params[col]:+.4f}{sig}  p={model.pvalues[col]:.4f}")
    except Exception as e:
        print(f"Interaction model failed: {e}")


In [ ]:
# --- Seasonal Visualization ---
# Line chart: elasticity by period per brand
brand_seasonal = df_seasonal[df_seasonal["brand"] != "ALL"].copy()

if len(brand_seasonal) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Line chart
    for brand in top_brands_s[:8]:  # top 8 for readability
        b_data = brand_seasonal[brand_seasonal["brand"] == brand].sort_values("period")
        if len(b_data) >= 2:
            axes[0].plot(b_data["period"], b_data["elasticity"], marker="o", label=brand, linewidth=2)
    axes[0].axhline(y=0, color="gray", linestyle="--", alpha=0.5)
    axes[0].set_title("Price Elasticity by Sub-Period (Top Brands)")
    axes[0].set_ylabel("Elasticity")
    axes[0].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    axes[0].tick_params(axis="x", rotation=15)

    # Heatmap
    pivot_s = brand_seasonal.pivot(index="brand", columns="period", values="elasticity")
    sns.heatmap(pivot_s, cmap="RdYlGn", center=0, annot=True, fmt=".2f",
                linewidths=0.5, ax=axes[1], cbar_kws={"label": "Elasticity"})
    axes[1].set_title("Seasonal Elasticity Heatmap")

    plt.tight_layout()
    plt.show()

    # Narrative: brands with largest seasonal shift
    brand_shift = brand_seasonal.groupby("brand")["elasticity"].agg(["min", "max"])
    brand_shift["range"] = brand_shift["max"] - brand_shift["min"]
    top_shifters = brand_shift.nlargest(5, "range")
    print("\n=== Brands with Largest Seasonal Elasticity Shift ===")
    for brand, r in top_shifters.iterrows():
        print(f"  {brand}: range = {r['range']:.3f} (min={r['min']:.3f}, max={r['max']:.3f})")
else:
    print("Insufficient data for seasonal brand-level visualization.")


## 19. Brand Price Sensitivity Rankings

Ranked table of brands by price elasticity with strategic recommendations.
- **Inelastic** (elasticity closer to 0): candidates for price **increases**
- **Elastic** (large negative elasticity): candidates for price **decreases** to capture volume

In [ ]:
# --- Brand Sensitivity Rankings with Recommendations ---
MIN_OBS_BRAND = 30

all_brand_elast = []
for brand, grp in df_elast.groupby("brand"):
    if len(grp) < MIN_OBS_BRAND:
        continue
    try:
        X = sm.add_constant(np.log(grp["offer_price"]))
        y = np.log1p(grp["qty_sold"])
        model = sm.OLS(y, X).fit()
        all_brand_elast.append({
            "brand": brand, "elasticity": model.params.iloc[-1],
            "SE": model.bse.iloc[-1], "p_value": model.pvalues.iloc[-1], "n_obs": len(grp),
        })
    except Exception:
        continue

df_brand_rank = pd.DataFrame(all_brand_elast).sort_values("elasticity")

# Add context columns from the full dataset
brand_stats = df.groupby("brand").agg(
    avg_te_margin=("te_margin", "mean"),
    avg_wm_margin=("walmart_margin", "mean"),
    total_qty=("qty_sold", "sum"),
    total_revenue=("revenue", "sum"),
    n_sku_nodes=("SKU_Node", "nunique"),
).reset_index()
df_brand_rank = df_brand_rank.merge(brand_stats, on="brand", how="left")

# Classify sensitivity tier
def _classify(e):
    if e < -1.5: return "Highly Elastic"
    if e < -0.8: return "Elastic"
    if e < -0.3: return "Unit Elastic"
    return "Inelastic"

df_brand_rank["sensitivity_tier"] = df_brand_rank["elasticity"].apply(_classify)

# Recommendation
def _recommend(e):
    if e < -0.8: return "Decrease price to capture volume"
    if e > -0.3: return "Increase price to capture margin"
    return "Monitor - near unit elastic"

df_brand_rank["recommendation"] = df_brand_rank["elasticity"].apply(_recommend)

# Display
print(f"Brands with valid elasticity estimates: {len(df_brand_rank)}")
display(df_brand_rank.style.format({
    "elasticity": "{:.4f}", "SE": "{:.4f}", "p_value": "{:.4f}",
    "avg_te_margin": "{:.1%}", "avg_wm_margin": "{:.1%}",
    "total_qty": "{:,.0f}", "total_revenue": "${:,.0f}",
}).background_gradient(subset=["elasticity"], cmap="RdYlGn"))

# Top candidates
increase = df_brand_rank[df_brand_rank["recommendation"].str.contains("Increase")].head(5)
decrease = df_brand_rank[df_brand_rank["recommendation"].str.contains("Decrease")].head(5)

print("\n=== TOP 5 PRICE INCREASE CANDIDATES (Inelastic) ===")
for _, r in increase.iterrows():
    print(f"  {r['brand']}: elasticity={r['elasticity']:.3f}, current TE margin={r['avg_te_margin']:.1%}, qty={r['total_qty']:,.0f}")

print("\n=== TOP 5 PRICE DECREASE CANDIDATES (Elastic) ===")
for _, r in decrease.iterrows():
    print(f"  {r['brand']}: elasticity={r['elasticity']:.3f}, current TE margin={r['avg_te_margin']:.1%}, qty={r['total_qty']:,.0f}")


## 20. Optimal Margin Targeting

For each brand, estimate the TE margin and Walmart margin that maximize revenue or profit.
Fit a quadratic relationship: `qty ~ margin + margin²` to capture the inverted-U shape.

In [ ]:
# --- Margin-Sales Relationship by Brand ---
top_brands_margin = df_brand_rank["brand"].head(20).tolist() if len(df_brand_rank) > 0 else top_brands_elast

margin_opt_results = []
df_margin = df[(df["qty_sold"] > 0) & (df["te_margin"].notna()) & (df["te_margin"].between(-0.5, 0.8))].copy()

for brand in top_brands_margin:
    b_data = df_margin[df_margin["brand"] == brand]
    if len(b_data) < 50:
        continue

    for margin_col, margin_name in [("te_margin", "TE Margin"), ("walmart_margin", "Walmart Margin")]:
        b_valid = b_data[b_data[margin_col].notna() & b_data[margin_col].between(-0.5, 0.8)]
        if len(b_valid) < 50:
            continue
        try:
            m = b_valid[margin_col].values
            m2 = m ** 2
            X = sm.add_constant(np.column_stack([m, m2]))
            y = b_valid["qty_sold"].values
            model = sm.OLS(y, X).fit()

            # Vertex of parabola: optimal = -b1 / (2*b2)
            b1, b2 = model.params[1], model.params[2]
            if b2 != 0:
                optimal = -b1 / (2 * b2)
                # Only report if optimal is in a reasonable range
                if -0.1 <= optimal <= 0.5:
                    margin_opt_results.append({
                        "brand": brand, "margin_type": margin_name,
                        "optimal_margin": optimal,
                        "current_avg": b_valid[margin_col].mean(),
                        "gap_pct": optimal - b_valid[margin_col].mean(),
                        "b1": b1, "b2": b2, "R2": model.rsquared,
                        "n_obs": len(b_valid), "is_concave": b2 < 0,
                    })
        except Exception:
            continue

df_margin_opt = pd.DataFrame(margin_opt_results)
if len(df_margin_opt) > 0:
    print("=== Optimal Margin Estimates (Quadratic Fit) ===")
    print("Note: 'is_concave=True' means the relationship has a peak (inverted-U).\n")
    display(df_margin_opt.style.format({
        "optimal_margin": "{:.1%}", "current_avg": "{:.1%}", "gap_pct": "{:+.1%}",
        "b1": "{:.4f}", "b2": "{:.4f}", "R2": "{:.4f}",
    }))

    # Flag brands where current margin is far from optimal
    big_gaps = df_margin_opt[(df_margin_opt["gap_pct"].abs() > 0.03) & df_margin_opt["is_concave"]]
    if len(big_gaps) > 0:
        print("\n=== Brands with >3pp Gap from Optimal Margin ===")
        for _, r in big_gaps.iterrows():
            direction = "increase" if r["gap_pct"] > 0 else "decrease"
            print(f"  {r['brand']} ({r['margin_type']}): optimal={r['optimal_margin']:.1%}, "
                  f"current={r['current_avg']:.1%} -> {direction} by {abs(r['gap_pct']):.1%}")
else:
    print("No brands with valid quadratic margin-sales fit in reasonable range.")


In [ ]:
# --- Profit-Maximizing Margin ---
profit_opt_results = []

for brand in top_brands_margin:
    b_data = df_margin[df_margin["brand"] == brand]
    if len(b_data) < 50:
        continue
    b_valid = b_data[b_data["te_margin"].notna() & b_data["cost_to_walmart"].notna()].copy()
    if len(b_valid) < 50:
        continue

    # Profit proxy: qty * te_margin * cost_to_walmart
    b_valid["profit_proxy"] = b_valid["qty_sold"] * b_valid["te_margin"] * b_valid["cost_to_walmart"]

    try:
        m = b_valid["te_margin"].values
        X = sm.add_constant(np.column_stack([m, m ** 2]))
        y = b_valid["profit_proxy"].values
        model = sm.OLS(y, X).fit()
        b1, b2 = model.params[1], model.params[2]
        if b2 != 0:
            optimal_profit = -b1 / (2 * b2)
            if -0.1 <= optimal_profit <= 0.5:
                profit_opt_results.append({
                    "brand": brand,
                    "profit_max_margin": optimal_profit,
                    "current_avg": b_valid["te_margin"].mean(),
                    "is_concave": b2 < 0,
                    "R2": model.rsquared,
                })
    except Exception:
        continue

df_profit_opt = pd.DataFrame(profit_opt_results)

# Compare revenue-maximizing vs profit-maximizing
if len(df_profit_opt) > 0 and len(df_margin_opt) > 0:
    te_rev = df_margin_opt[df_margin_opt["margin_type"] == "TE Margin"][["brand", "optimal_margin"]].rename(
        columns={"optimal_margin": "rev_max_margin"})
    comparison = te_rev.merge(df_profit_opt[["brand", "profit_max_margin", "current_avg"]], on="brand")
    comparison["rev_vs_profit_gap"] = comparison["rev_max_margin"] - comparison["profit_max_margin"]

    print("=== Revenue-Maximizing vs Profit-Maximizing TE Margin ===\n")
    display(comparison.style.format({
        "rev_max_margin": "{:.1%}", "profit_max_margin": "{:.1%}",
        "current_avg": "{:.1%}", "rev_vs_profit_gap": "{:+.1%}",
    }))

    # Scatter plot
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.scatter(comparison["rev_max_margin"], comparison["profit_max_margin"], s=80, alpha=0.7)
    for _, r in comparison.iterrows():
        ax.annotate(r["brand"], (r["rev_max_margin"], r["profit_max_margin"]),
                    fontsize=8, ha="left", va="bottom")
    lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lims, lims, "k--", alpha=0.3, label="Equal")
    ax.set_xlabel("Revenue-Maximizing TE Margin")
    ax.set_ylabel("Profit-Maximizing TE Margin")
    ax.set_title("Revenue vs Profit Optimal Margins by Brand")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Insufficient data for profit-maximizing comparison.")


## 21. What-If Price Change Simulation

Simulate gradual price changes (-5% to +5%) for each brand using estimated elasticity.
- **Elastic brands:** simulate price decreases to capture volume
- **Inelastic brands:** simulate price increases to capture margin

Identify diminishing returns.

In [ ]:
# --- Simulation Engine ---
if len(df_brand_rank) > 0:
    pct_changes = np.arange(-0.05, 0.06, 0.01)

    sim_results = []
    for _, br in df_brand_rank.iterrows():
        brand = br["brand"]
        elast = br["elasticity"]
        # Baseline from actual data
        b_data = df[(df["brand"] == brand) & (df["qty_sold"] > 0)]
        if len(b_data) < 30:
            continue
        base_price = b_data["offer_price"].mean()
        base_qty = b_data["qty_sold"].mean()
        base_revenue = base_qty * base_price
        base_cost = b_data["cost_to_walmart"].mean() if "cost_to_walmart" in b_data.columns else base_price * 0.85
        base_profit = base_qty * (base_price - base_cost)

        for pct in pct_changes:
            new_price = base_price * (1 + pct)
            # Constant elasticity model: Q_new = Q_base * (P_new / P_base) ^ elasticity
            if base_price > 0:
                projected_qty = base_qty * (new_price / base_price) ** elast
            else:
                projected_qty = base_qty
            projected_revenue = projected_qty * new_price
            projected_profit = projected_qty * (new_price - base_cost)

            sim_results.append({
                "brand": brand, "pct_change": pct,
                "new_price": new_price, "projected_qty": projected_qty,
                "projected_revenue": projected_revenue, "projected_profit": projected_profit,
                "revenue_change_pct": (projected_revenue - base_revenue) / base_revenue if base_revenue > 0 else 0,
                "profit_change_pct": (projected_profit - base_profit) / base_profit if base_profit > 0 else 0,
                "base_revenue": base_revenue, "base_profit": base_profit,
            })

    df_sim = pd.DataFrame(sim_results)
    print(f"Simulation complete: {df_sim['brand'].nunique()} brands, {len(pct_changes)} price points each")
    print(f"Total simulation rows: {len(df_sim):,}")
else:
    df_sim = pd.DataFrame()
    print("No brand elasticity data available for simulation.")


In [ ]:
# --- Visualization & Diminishing Returns ---
if len(df_sim) > 0:
    elastic_brands = df_brand_rank[df_brand_rank["elasticity"] < -0.8]["brand"].head(5).tolist()
    inelastic_brands = df_brand_rank[df_brand_rank["elasticity"] > -0.3]["brand"].tail(5).tolist()

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Elastic brands: revenue impact of price decreases
    for brand in elastic_brands:
        b_sim = df_sim[(df_sim["brand"] == brand) & (df_sim["pct_change"] <= 0)].sort_values("pct_change")
        if len(b_sim) > 0:
            axes[0].plot(b_sim["pct_change"] * 100, b_sim["revenue_change_pct"] * 100,
                        marker="o", label=brand, linewidth=2)
    axes[0].axhline(y=0, color="gray", linestyle="--", alpha=0.5)
    axes[0].set_title("Elastic Brands: Revenue Impact of Price Decrease")
    axes[0].set_xlabel("Price Change (%)")
    axes[0].set_ylabel("Revenue Change (%)")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Inelastic brands: profit impact of price increases
    for brand in inelastic_brands:
        b_sim = df_sim[(df_sim["brand"] == brand) & (df_sim["pct_change"] >= 0)].sort_values("pct_change")
        if len(b_sim) > 0:
            axes[1].plot(b_sim["pct_change"] * 100, b_sim["profit_change_pct"] * 100,
                        marker="o", label=brand, linewidth=2)
    axes[1].axhline(y=0, color="gray", linestyle="--", alpha=0.5)
    axes[1].set_title("Inelastic Brands: Profit Impact of Price Increase")
    axes[1].set_xlabel("Price Change (%)")
    axes[1].set_ylabel("Profit Change (%)")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Diminishing returns: find sweet spot per brand
    sweet_spot = []
    for brand in df_sim["brand"].unique():
        b_sim = df_sim[df_sim["brand"] == brand].sort_values("pct_change")
        b_sim["marginal_profit"] = b_sim["projected_profit"].diff() / (b_sim["pct_change"].diff() * b_sim["base_profit"].iloc[0]) if b_sim["base_profit"].iloc[0] != 0 else 0

        elast = df_brand_rank[df_brand_rank["brand"] == brand]["elasticity"].iloc[0]
        if elast < -0.8:  # elastic -> look at decreases
            decreases = b_sim[b_sim["pct_change"] < 0].sort_values("pct_change", ascending=False)
            best_pct = decreases.loc[decreases["revenue_change_pct"].idxmax(), "pct_change"] if len(decreases) > 0 else 0
            best_rev = decreases["revenue_change_pct"].max() if len(decreases) > 0 else 0
            sweet_spot.append({"brand": brand, "strategy": "Decrease price",
                               "recommended_pct": best_pct, "expected_revenue_chg": best_rev})
        elif elast > -0.3:  # inelastic -> look at increases
            increases = b_sim[b_sim["pct_change"] > 0].sort_values("pct_change")
            best_pct = increases.loc[increases["profit_change_pct"].idxmax(), "pct_change"] if len(increases) > 0 else 0
            best_prof = increases["profit_change_pct"].max() if len(increases) > 0 else 0
            sweet_spot.append({"brand": brand, "strategy": "Increase price",
                               "recommended_pct": best_pct, "expected_profit_chg": best_prof})

    df_sweet = pd.DataFrame(sweet_spot)
    if len(df_sweet) > 0:
        print("\n=== Recommended Price Changes (Sweet Spot) ===")
        display(df_sweet.style.format({
            "recommended_pct": "{:+.0%}",
            "expected_revenue_chg": "{:+.1%}",
            "expected_profit_chg": "{:+.1%}",
        }, na_rep="-"))
else:
    print("No simulation data available.")


## 22. Actionable Pricing Strategy by Brand-State Segment

Capstone section combining elasticity, treatment effects, margin analysis, and simulation
into per-segment recommendations.

In [ ]:
# --- Build Master Strategy Table ---
strategy_rows = []

# Use brand-state elasticity from §16
if len(df_bs_elast) > 0 and len(df_brand_rank) > 0:
    bs_valid = df_bs_elast.dropna(subset=["elasticity"]).copy()

    for _, r in bs_valid.iterrows():
        brand, state = r["brand"], r["State"]
        elast = r["elasticity"]
        n = r["n"]
        pval = r["p_value"]

        # Get brand-level info
        br_info = df_brand_rank[df_brand_rank["brand"] == brand]
        avg_te = br_info["avg_te_margin"].iloc[0] if len(br_info) > 0 else np.nan
        avg_wm = br_info["avg_wm_margin"].iloc[0] if len(br_info) > 0 else np.nan

        # Get treatment effect if available
        att = np.nan
        if len(df_het) > 0:
            brand_het = df_het[(df_het["segment_type"] == "Brand") & (df_het["segment_value"] == brand)]
            if len(brand_het) > 0:
                att = brand_het["ATT"].iloc[0]

        # Get optimal margin if available
        opt_margin = np.nan
        if len(df_margin_opt) > 0:
            opt = df_margin_opt[(df_margin_opt["brand"] == brand) & (df_margin_opt["margin_type"] == "TE Margin")]
            if len(opt) > 0:
                opt_margin = opt["optimal_margin"].iloc[0]

        # Recommendation logic
        if elast < -0.8:
            action = "Decrease"
            rec_pct = max(-0.05, elast * 0.02)  # conservative: small fraction of elasticity
        elif elast > -0.3:
            action = "Increase"
            rec_pct = min(0.05, abs(elast) * 0.03)
        else:
            action = "Hold"
            rec_pct = 0.0

        # Projected impact (using constant elasticity)
        expected_qty_pct = (1 + rec_pct) ** elast - 1 if rec_pct != 0 else 0
        expected_rev_pct = ((1 + rec_pct) * (1 + expected_qty_pct)) - 1

        # Confidence
        if pval < 0.01 and n > 200:
            confidence = "High"
        elif pval < 0.05 and n > 50:
            confidence = "Medium"
        else:
            confidence = "Low"

        strategy_rows.append({
            "brand": brand, "State": state, "elasticity": elast,
            "treatment_effect": att, "optimal_te_margin": opt_margin,
            "current_te_margin": avg_te, "current_wm_margin": avg_wm,
            "action": action, "recommended_change": rec_pct,
            "expected_qty_impact": expected_qty_pct,
            "expected_revenue_impact": expected_rev_pct,
            "confidence": confidence, "n_obs": n, "p_value": pval,
        })

df_strategy = pd.DataFrame(strategy_rows)
if len(df_strategy) > 0:
    # Sort by expected revenue impact
    df_strategy = df_strategy.sort_values("expected_revenue_impact", ascending=False)
    print(f"Strategy table: {len(df_strategy)} brand-state segments")
    display(df_strategy.head(20).style.format({
        "elasticity": "{:.3f}", "treatment_effect": "{:+.4f}",
        "optimal_te_margin": "{:.1%}", "current_te_margin": "{:.1%}", "current_wm_margin": "{:.1%}",
        "recommended_change": "{:+.1%}", "expected_qty_impact": "{:+.1%}",
        "expected_revenue_impact": "{:+.1%}", "p_value": "{:.4f}",
    }, na_rep="-").background_gradient(subset=["expected_revenue_impact"], cmap="RdYlGn"))
else:
    print("Insufficient data to build strategy table.")


In [ ]:
# --- Narrative Summary ---
if len(df_strategy) > 0:
    print("=" * 70)
    print("ACTIONABLE PRICING RECOMMENDATIONS")
    print("=" * 70)

    # Top 10 highest-impact
    top10 = df_strategy.head(10)
    print("\n--- TOP 10 HIGHEST REVENUE IMPACT ---\n")
    for i, (_, r) in enumerate(top10.iterrows(), 1):
        sig = f"[{r['confidence']} confidence]"
        print(f"{i}. **{r['brand']} in {r['State']}** (elasticity={r['elasticity']:.3f}, n={r['n_obs']:,})")
        print(f"   {r['action']} price by {abs(r['recommended_change']):.0%} -> "
              f"Expected revenue: {r['expected_revenue_impact']:+.1%}, "
              f"Expected qty: {r['expected_qty_impact']:+.1%}  {sig}")

    # Quick wins: high confidence, positive revenue impact
    quick_wins = df_strategy[(df_strategy["confidence"] == "High") &
                              (df_strategy["expected_revenue_impact"] > 0)].head(5)
    if len(quick_wins) > 0:
        print("\n--- QUICK WINS (High Confidence, Positive Revenue) ---\n")
        for _, r in quick_wins.iterrows():
            print(f"  {r['brand']} in {r['State']}: {r['action']} {abs(r['recommended_change']):.0%} "
                  f"-> +{r['expected_revenue_impact']:.1%} revenue (n={r['n_obs']:,})")

    # Caution: low confidence or counter-intuitive
    caution = df_strategy[df_strategy["confidence"] == "Low"].head(5)
    if len(caution) > 0:
        print("\n--- CAUTION (Low Confidence - Need More Data) ---\n")
        for _, r in caution.iterrows():
            print(f"  {r['brand']} in {r['State']}: elasticity={r['elasticity']:.3f} "
                  f"(p={r['p_value']:.3f}, n={r['n_obs']:,}) — insufficient evidence")
else:
    print("No strategy recommendations available.")


## 23. Extended Executive Summary

In [ ]:
# --- Extended Executive Summary ---
print("=" * 70)
print("EXTENDED EXECUTIVE SUMMARY")
print(f"Analysis Window: {START_DATE} to {END_DATE}")
print(f"Dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")
print("=" * 70)

# New features
print("\n1. NEW FEATURES")
print(f"   - Tire size coverage: {df['tire_size'].notna().mean():.1%}")
print(f"   - MAP-governed tires: {df['is_MAP_tire'].mean():.1%}")
if df["tire_diameter"].notna().any():
    print(f"   - Most common diameters: {df['tire_diameter'].value_counts().head(5).to_dict()}")

# State elasticity
print("\n2. STATE-LEVEL PRICE SENSITIVITY")
if len(df_state_elast) > 0:
    most_sens = df_state_elast.iloc[0]
    least_sens = df_state_elast.iloc[-1]
    print(f"   - Most price-sensitive state: {most_sens['State']} (elasticity={most_sens['elasticity']:.3f})")
    print(f"   - Least price-sensitive state: {least_sens['State']} (elasticity={least_sens['elasticity']:.3f})")
    print(f"   - States analyzed: {len(df_state_elast)}")

# Brand-State highlights
print("\n3. BRAND-STATE HIGHLIGHTS")
if len(df_bs_elast) > 0:
    bs_valid = df_bs_elast.dropna(subset=["elasticity"])
    print(f"   - Brand-State segments analyzed: {len(bs_valid)}")
    if len(bs_valid) > 0:
        top = bs_valid.nsmallest(3, "elasticity")
        for _, r in top.iterrows():
            print(f"   - {r['brand']} in {r['State']}: elasticity={r['elasticity']:.3f} (highly sensitive)")

# Treatment effects
print("\n4. HETEROGENEOUS TREATMENT EFFECTS")
if len(df_het) > 0:
    map_effects = df_het[df_het["segment_type"] == "MAP Status"]
    if len(map_effects) > 0:
        for _, r in map_effects.iterrows():
            print(f"   - {r['segment_value']}: ATT={r['ATT']:+.4f}{r['sig']}")

# Seasonal
print("\n5. SEASONAL PRICE SENSITIVITY")
if len(df_seasonal) > 0:
    overall = df_seasonal[df_seasonal["brand"] == "ALL"]
    for _, r in overall.iterrows():
        print(f"   - {r['period']}: overall elasticity={r['elasticity']:.4f}")

# Strategy highlights
print("\n6. TOP RECOMMENDATIONS")
if len(df_strategy) > 0:
    n_increase = (df_strategy["action"] == "Increase").sum()
    n_decrease = (df_strategy["action"] == "Decrease").sum()
    n_hold = (df_strategy["action"] == "Hold").sum()
    print(f"   - {n_increase} segments: increase price")
    print(f"   - {n_decrease} segments: decrease price")
    print(f"   - {n_hold} segments: hold current price")
    high_conf = df_strategy[df_strategy["confidence"] == "High"]
    print(f"   - High-confidence recommendations: {len(high_conf)}")

print("\n" + "=" * 70)
print("END OF EXTENDED ANALYSIS")
print("=" * 70)


## 24. Save & Cleanup

In [ ]:
# Save the extended dataset (with new features)
project_root = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
output_dir = os.path.join(project_root, "outputs")
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, f"correlation_dataset_{END_DATE}.parquet")
try:
    df.to_parquet(output_path, index=False)
    print(f"Dataset saved to {output_path}")
except Exception as e:
    output_path = output_path.replace(".parquet", ".csv")
    df.to_csv(output_path, index=False)
    print(f"Parquet failed ({e}), saved as CSV: {output_path}")

print(f"Shape: {df.shape}")

# Cleanup
try:
    loader.close()
except NameError:
    pass  # loader not available in cached mode
print("Done.")
